# 광각(wide) 재학습 — wide_v6, 6클래스 태극기 깃발 포함

4종 도형 + `fruit_photo_cube` + `arrival` YOLOv8n. 기존 `models/wide.pt`를 대체할 광각 카메라용 모델입니다.

- 업로드: `dataset_wide_flag_v6.zip` (내부 루트 `wide_dataset/{images,labels}`)
- 클래스 순서: `cube`, `octahedron`, `dodecahedron`, `icosahedron`, `fruit_photo_cube`, `arrival`
- 광각은 물체가 작게 보여 `imgsz=1280`으로 학습합니다.
- 결과 `best.pt`를 젯슨의 `models/wide.pt`로 배포합니다.

In [ ]:
# [셀1] 업로드 + 압축해제
from google.colab import files
import os, shutil
up = files.upload()
ZIP = next(iter(up))
shutil.rmtree('/content/wide_dataset', ignore_errors=True)
!unzip -o -q "$ZIP" -d /content
print('uploaded:', ZIP, '-> extracted:', sorted(os.listdir('/content/wide_dataset')))

In [ ]:
# [셀2] train/val 분리 + data_colab.yaml 생성
import os, glob, random, shutil, yaml
random.seed(0)
ROOT = '/content/wide_dataset'
CLASSES = ['cube', 'octahedron', 'dodecahedron', 'icosahedron', 'fruit_photo_cube', 'arrival']
HELD_OUT_VAL = True
VAL_FRAC = 0.15
lbl = lambda p: '/content/wide_dataset/labels/' + os.path.splitext(os.path.basename(p))[0] + '.txt'
pairs = [(i, lbl(i)) for i in sorted(glob.glob('/content/wide_dataset/images/*')) if os.path.exists(lbl(i))]
print('pairs:', len(pairs), '(빈 라벨=배경음성 포함)')
if HELD_OUT_VAL:
    random.shuffle(pairs); n = int(len(pairs) * VAL_FRAC)
    for split, items in [('train', pairs[n:]), ('val', pairs[:n])]:
        for s in ('images', 'labels'):
            os.makedirs(f'/content/wide_dataset/{split}/{s}', exist_ok=True)
        for img, lb in items:
            shutil.copy(img, f'/content/wide_dataset/{split}/images/')
            shutil.copy(lb, f'/content/wide_dataset/{split}/labels/')
    print('train', len(pairs) - n, '/ val', n)
    data = dict(path=ROOT, train='train/images', val='val/images')
else:
    data = dict(path=ROOT, train='images', val='images')
data.update(nc=len(CLASSES), names=CLASSES)
yaml.safe_dump(data, open('/content/wide_dataset/data_colab.yaml', 'w'))
print(open('/content/wide_dataset/data_colab.yaml').read())

In [ ]:
# [셀3] 학습 — 광각용 imgsz=1280
!pip -q install ultralytics
from ultralytics import YOLO
YOLO('yolov8n.pt').train(data='/content/wide_dataset/data_colab.yaml',
    epochs=100, imgsz=1280, batch=8, patience=30, name='wide_v6_flag')

In [ ]:
# [셀4] best.pt 내려받기
from google.colab import files
files.download('runs/detect/wide_v6_flag/weights/best.pt')